# GDN-3 на Colab (TPU v5e)
Рантайм: **TPU v5e**. Порядок: установка → распаковка → тесты → скорость → MQAR → LM. Если тест падает — дальше не идти (docs/EXPERIMENTS.md).

In [ ]:
!pip -q install -U 'jax[tpu]' optax -f https://storage.googleapis.com/jax-releases/libtpu_releases.html
import jax; print(jax.__version__, jax.devices())

In [ ]:
# загрузите gdn3_pack.zip (слева Files) или смонтируйте Drive, затем:
!unzip -q -o /content/gdn3_pack.zip -d /content
%cd /content/gdn3_pack

In [ ]:
# ядро atomic_gdn2
!git clone --depth 1 https://github.com/Akseleu-J/atomic-ops.git /content/atomic-ops
import sys; sys.path.insert(0, '/content/gdn3_pack')
from gdn3.backend import ensure_atomic_path
print('atomic_gdn2 found:', ensure_atomic_path('/content/atomic-ops'))

In [ ]:
# M0: математика (numpy) и тесты JAX на эталонном бэкенде
!python -m tests.test_numpy_ref
!python -m tests.run_all

In [ ]:
# M0: Pallas-ядро == эталон. Если FAIL — пришлите вывод целиком.
!python -m tests.run_all --atomic

In [ ]:
# механизм-бенчмарк (numpy, секунды)
!python -m bench.mechanism_np

In [ ]:
# M1: скорость
!python -m bench.speed --backend atomic --L 4096 --seg 128 256 --bank_every 2 1

In [ ]:
# M2 smoke (300 шагов) — убедиться, что всё работает
!python -m train.train_mqar --model gdn3 --steps 300 --log_every 50 --seg 128 --L 4096 --out runs/smoke --eval_Ps 64 128 --eval_Ls 4096

In [ ]:
# M2 полный прогон (3 модели x 3 seed). ~ часы; можно запускать по одной.
for m in ['gdn2','gdn2c','gdn3']:
    for s in [0,1,2]:
        !python -m train.train_mqar --model {m} --seed {s} --L 4096 --seg 128 --steps 6000 --out runs/mqar_{m}_s{s}

In [ ]:
# M3: LM на enwik8
!python -m train.train_lm --model gdn2c --data enwik8 --L 8192 --bsz 4 --steps 20000 --out runs/lm_gdn2c_s0
!python -m train.train_lm --model gdn3  --data enwik8 --L 8192 --bsz 4 --steps 20000 --seg 128 --out runs/lm_gdn3_s0

In [ ]:
# сводка из логов
import json, glob
for f in sorted(glob.glob('runs/*/log.jsonl')):
    rows=[json.loads(l) for l in open(f)]
    fin=[r for r in rows if r.get('final')]; val=[r for r in rows if 'val' in r]
    print(f, '| P*:', fin[-1]['Pstar'] if fin else '-', '| last val:', val[-1]['val'] if val else '-')